In [0]:
import pandas as pd

gold_df = spark.table("causal_project.gold_household").toPandas()

gold_encoded = gold_df.copy()

gold_encoded["age_group"] = gold_encoded["age_group"].map({
    "Age Group1": 1,
    "Age Group2": 2,
    "Age Group3": 3,
    "Age Group4": 4,
    "Age Group5": 5,
    "Age Group6": 6
})

gold_encoded["income_level"] = gold_encoded["income_level"].str.replace(
    "Level", "", regex=False
).astype(int)

gold_encoded["household_size"] = gold_encoded["household_size"].map({
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5+": 5
})

gold_encoded["kid_count"] = gold_encoded["kid_count"].map({
    "None/Unknown": 0,
    "1": 1,
    "2": 2,
    "3+": 3
})

gold_encoded["home_ownership"] = gold_encoded["home_ownership"].map({
    "Renter": 0,
    "Probable Renter": 1,
    "Unknown": 2,
    "Probable Owner": 3,
    "Homeowner": 4
})

gold_encoded["marital_status"] = gold_encoded["marital_status"].map({
    "X": 0,
    "Y": 1,
    "Z": 2
})

In [0]:

import numpy as np
gold_encoded['log_outcome'] = np.log1p(gold_df['avg_daily_campaign_spend'])

In [0]:

discovery_cols = [
  'treatment',
  'log_outcome',
  'pre_avg_weekly_spend',
  'pre_avg_weekly_trips', 
  'pre_coupon_usage_rate',
  'pre_loyalty_card_rate',
  'pre_department_breadth',
  'pre_active_weeks',
  'clean_window_days',
  'age_group',
  'marital_status',
  'income_level',
  'household_size',
  'home_ownership',
  'kid_count'
]

discovery_df = gold_encoded[discovery_cols]

In [0]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

data = discovery_df.to_numpy().astype(float)
demo = [  'age_group',
  'marital_status',
  'income_level',
  'household_size',
  'home_ownership',
  'kid_count'
]
pre = [
  'pre_avg_weekly_spend',
  'pre_avg_weekly_trips', 
  'pre_coupon_usage_rate',
  'pre_loyalty_card_rate',
  'pre_department_breadth',
  'pre_active_weeks']

during = ['treatment','clean_window_days']

post = ['log_outcome']

TIER_MAP = {
    **{c:0 for c in demo},
    **{c:1 for c in pre},
    **{c:2 for c in during},
    **{c:3 for c in post}
}
def build_background_knowledge(col_names):
    nodes = [GraphNode(col) for col in col_names]
    col_to_node = {col: nodes[i] for i, col in enumerate(col_names)}
    
    bk = BackgroundKnowledge()

    for col, tier in TIER_MAP.items():
        if col in col_to_node:
            bk.add_node_to_tier(col_to_node[col], tier)

    return bk
background = build_background_knowledge(discovery_df.columns.to_list())

cg = pc(data, alpha=0.05, indep_test=fisherz,node_names= discovery_df.columns.to_list(),background_knowledge=background)

print(cg.G)

## Causal Discovery Results : PC Algorithm with Temporal Background Knowledge

### What the algorithm found
The PC algorithm recovered a sparse DAG across 15 variables using 
Fisher-Z conditional independence tests (α=0.05, n=760).

Key findings:

**Behavioral cluster**: A clear causal chain runs through the 
pre-campaign features: active weeks and coupon usage drive loyalty 
card rate, which drives spend and trips, which drives department 
breadth. This is consistent with domain knowledge: households that 
shop more frequently naturally accumulate broader category exposure 
and higher spend.

**Outcome**: Only `pre_avg_weekly_spend` and `pre_loyalty_card_rate` 
have direct edges to `log_outcome`. This confirms these two variables 
are the strongest confounders and must be included in W (controls).

**Demographic cluster**: Marital status, income, home ownership and 
kid count form a partially undirected subgraph (the algorithm could 
not determine causal direction among them). Treated as effect modifiers 
in X rather than confounders in W.

---

### Domain Knowledge Overrides Applied to PC Output

Three changes were made to the algorithmically recovered graph before 
use in DoWhy and EconML:

**Override 1 — Removed `treatment → clean_window_days`**

The PC algorithm recovered an edge from treatment to campaign window 
length. This is a data artifact — TypeA campaign windows were longer 
on average than TypeBC windows (SMD = 1.109, mean 43.9 vs 30.2 days), 
creating a spurious statistical association. Campaign window length 
is determined at the time of campaign design, not caused by which 
household receives the campaign. This edge is removed.
`clean_window_days` is retained in W as a technical control variable 
and its arrow into treatment is preserved — window length is 
associated with treatment assignment, but treatment does not cause 
window length.

**Override 2 — Added `treatment → log_outcome`**

The PC algorithm found no direct edge from treatment to outcome. 
This does not mean no causal effect exists. At n=760, the Fisher-Z 
test may lack statistical power to detect a weak or heterogeneous 
effect — particularly one where individual effects vary substantially 
in direction across subgroups (as confirmed by the CATE distribution 
in notebook 06). The edge is added based on the research question: 
campaign assignment is hypothesized to affect spend, and EconML 
estimates this effect directly without requiring the PC algorithm 
to confirm it first.

**Override 3 — Added behavioral features as parents of treatment**

The PC algorithm found no edges from pre-campaign behavioral features 
into treatment. With n=760 and Fisher-Z at α=0.05, the test lacked 
power to recover these relationships as graph edges. However, EDA 
confirmed meaningful imbalance on all behavioral features between 
TypeA and TypeBC households (SMDs ranging from -0.099 to -0.315), 
providing direct evidence of confounding. These variables are added 
as parents of treatment in the DoWhy graph, consistent with their 
role in the EconML W matrix and the confounding story established 
in 04_eda.

---

### Final graph used for DoWhy refutation
The graph used in notebook 07 reflects the PC algorithm output with 
the three domain knowledge overrides above applied. All overrides 
are documented and justified — none are introduced without empirical 
or theoretical support.

### Conclusion
The DAG supports the confounding story established in EDA. Pre-campaign 
behavioral features are causally upstream of the outcome and correlated 
with treatment assignment — confirming that causal inference methods 
are necessary. Proceeding to CausalForestDML estimation.